In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "fitbit_prostate_cancer" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
dataset_31237560_fitbit_sleep_daily_summary_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (4163261, 200970)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                sex_at_birth_concept_id IN (45878463) ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_sleep_daily_summary_31237560_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_sleep_daily_summary_31237560",
  "fitbit_sleep_daily_summary_31237560_*.csv")
message(str_glue('The data will be written to {fitbit_sleep_daily_summary_31237560_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_31237560_fitbit_sleep_daily_summary_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_sleep_daily_summary_31237560_path,
  destination_format = "CSV")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_31237560/fitbit_sleep_daily_summary_31237560_*.csv. Use this path when reading the data into your notebooks in the future.



In [2]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_sleep_daily_summary_31237560_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_sleep <- read_bq_export_from_workspace_bucket(fitbit_sleep_daily_summary_31237560_path)

dim(pca_sleep)

head(pca_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260428/fitbit_sleep_daily_summary_31237560/fitbit_sleep_daily_summary_31237560_000000000000.csv.



[1] 382682     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2657569,2022-02-08,true,180,158,0,22,NA,5,133,20,22
2657569,2022-05-09,false,181,158,0,23,NA,0,158,0,23
2657569,2023-01-30,true,182,159,0,23,NA,22,116,21,23
2657569,2022-11-10,false,184,165,0,19,NA,21,135,9,19
2657569,2022-12-16,true,184,154,0,30,NA,15,130,9,30


In [3]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded



In [4]:
colnames(pca_sleep)[2] <- 'date'
pca_sleep_main <- pca_sleep |>
  filter(is_main_sleep == "true")

pca_sleep_nap <- pca_sleep |>
  filter(is_main_sleep == "false")

pca_sleep_sum <- sleep_sum_main_nap_func(df = pca_sleep)

In [11]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Prostate/sleep/', df = pca_sleep_sum)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  63.5   1st Qu.:  88.0  
 Median : 100.0   Median : 124.0  
 Mean   : 526.6   Mean   : 842.6  
 3rd Qu.: 835.5   3rd Qu.:1448.0  
 Max.   :3466.0   Max.   :4443.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1002338   Min.   :   1.0   Min.   :2011-08-03   Min.   :2015-03-31  
 1st Qu.:1878468   1st Qu.:  63.5   1st Qu.:2019-01-15   1st Qu.:2023-09-09  
 Median :3188701   Median : 100.0   Median :2023-03-02   Median :2023-10-01  
 Mean   :4166346   Mean   : 526.6   Mean   :2021-03-05   Mean   :2023-06-25  
 3rd Qu.:6321677   3rd Qu.: 835.5   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9962965   Max.   :3466.0   Max.   :2023-09-24   Max.   :2023-10-01  
 followup_days     missing_days   pct_missing        compliance_tier
 Min.   :   1.0   Min.   :   0   Min.   : 0.000   High (>90%):233   
 1st Qu.:  88.0   1st Qu.:   8   1st Qu.: 5.098   80-90%     : 92   
 Median : 124.0   Median :  39   Median :18.438   70-80%     : 75   
 Mean   : 842.6   Mean   : 316   Mean   :29.435   60-70%     : 48   
 3rd Qu.:1448.0   3rd Qu.: 250   3rd Qu.:49.963   50-60%     : 22   
 Max.   :4443.0   Max.   :3622   Max.   

NULL

In [7]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Prostate',
    df = pca_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Prostate',
    df = pca_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep   627 360.   82.9  375.  320.  419.  203.  464. 
2 minute_awake    627  51.2  16.9   51.6  41.1  61.7  21.2  76.1
3 minute_deep     611  47.3  14.5   46.8  37.3  56.6  26.0  72.3
4 minute_in_bed   627 412.   94.2  429.  368.  477.  227.  531. 
5 minute_light    611 270.   48.8  272.  237.  303.  190.  347. 
6 minute_rem      611  62.1  20.6   61.8  47.1  76.7  29.9  98.1
7 minute_wake     611  57.5  12.3   56.8  49.6  65.2  38.5  78.7
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep   601  95.9 22.7   95.2   82.5  107.   63.4  129. 
2 minute_awake    601  10.2  8.09   8.14   5.67  11.8   2.78  28.6
3 minute_deep     386  20.2 11.6   19.5   12.7   27     0     40  
4 minute_in_bed   601 107.  23.9  106.    92

In [8]:
head(pca_sleep_sum)
patient_stats_sum <- sleep_prep_stats(df = pca_sleep_sum, path.out =  '/home/jupyter/workspaces/result/Prostate/', cohort = 'Prostate', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1002338,2014-06-01,TRUE,FALSE,1,411,386,17,14,0,⋯,0,0,411,386,17,0,0,0,0,93.91727
1002338,2014-06-02,TRUE,FALSE,1,385,365,8,12,0,⋯,0,4,385,365,8,0,0,0,0,94.80519
1002338,2014-06-03,TRUE,FALSE,1,220,207,3,5,0,⋯,0,0,220,207,3,0,0,0,0,94.09091
1002338,2014-06-05,TRUE,FALSE,1,454,419,18,23,0,⋯,0,0,454,419,18,0,0,0,0,92.29075
1002338,2014-06-06,TRUE,FALSE,1,433,391,33,30,0,⋯,0,3,433,391,33,0,0,0,0,90.30023
1002338,2014-06-07,TRUE,FALSE,1,420,393,18,17,0,⋯,0,0,420,393,18,0,0,0,0,93.57143


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3     P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep      627 375.  80.7   389.  339.  428.  223.   479. 
2 minute_awake       627  52.7 17.4    53.0  42.5  62.9  21.3   79.4
3 minute_deep        627  37.2 18.4    38.1  24.3  51.1   2.48  65.0
4 minute_in_bed      627 429.  91.9   441.  389.  489.  247.   548. 
5 minute_light       627 210.  87.7   229.  156.  275.   24.1  325. 
6 minute_rem         627  49.0 25.1    49.6  30.4  67.5   2.35  88.6
7 minute_wake        627  45.0 19.6    48.0  33.7  58.5   4.43  73.6
8 sleep_efficiency   627  87.7  3.50   88.0  86.3  89.4  83.3   92.8


In [10]:
miss_gap_func(df = pca_sleep_sum, path.out =  '/home/jupyter/workspaces/result/Prostate/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0     2.0    13.0   130.2    52.0  3245.0 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        608         97.0         19         3.03
2         7        390         62.2        237        37.8 
3        14        294         46.9        333        53.1 
4        30        220         35.1        407        64.9 
5        60        146         23.3        481        76.7 


NULL

In [8]:
path.out = '/home/jupyter/workspaces/Data/'
cohort = 'Prostate'
df = pca_activity_sex
key_metrics <- c("steps", "calories_out", "sedentary_minutes",
                    "lightly_active_minutes", "fairly_active_minutes",
                    "very_active_minutes", "sex_at_birth")

    patient_stats <- df |>
    group_by(person_id) |>
    summarise(
        n_obs = n(),
        across(all_of(key_metrics), list(
        mean = ~mean(.x, na.rm = TRUE),
        sd   = ~sd(.x,   na.rm = TRUE),
        cv   = ~ifelse(mean(.x, na.rm = TRUE) > 0,
                        sd(.x, na.rm = TRUE) / mean(.x, na.rm = TRUE) * 100, NA)
        ), .names = "{.col}__{.fn}"),
        .groups = "drop"
    )
    patient_stats$cohort <- cohort
    write.table(patient_stats, paste0(path.out, cohort,'_activity_summary_stats.txt'), sep = '\t', quote = F, row.names = F)

    # ── 2. Summary table: distribution of patient means ────────────────────────
    mean_cols <- paste0(key_metrics, "__mean")

    cohort_summary <- patient_stats |>
    select(all_of(mean_cols)) |>
    pivot_longer(everything(), names_to = "metric", values_to = "value") |>
    mutate(metric = str_remove(metric, "__mean")) |>
    group_by(metric) |>
    summarise(
        N        = sum(!is.na(value)),
        Mean     = mean(value,              na.rm = TRUE),
        SD       = sd(value,                na.rm = TRUE),
        Median   = median(value,            na.rm = TRUE),
        Q1       = quantile(value, 0.25,    na.rm = TRUE),
        Q3       = quantile(value, 0.75,    na.rm = TRUE),
        P5       = quantile(value, 0.05,    na.rm = TRUE),
        P95      = quantile(value, 0.95,    na.rm = TRUE),
        .groups  = "drop"
    )
    print(cohort_summary, n = Inf)

Warning message:
“There were 1881 warnings in `summarise()`.
The first warning was:
ℹ In argument: `across(...)`.
ℹ In group 1: `person_id = 1002338`.
Caused by warning in `mean.default()`:
! argument is not numeric or logical: returning NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1880 remaining warnings.”


# A tibble: 7 × 9
  metric                  N   Mean     SD Median      Q1     Q3       P5     P95
  <chr>               <int>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>    <dbl>   <dbl>
1 calories_out          627 2529.   438.  2507.  2239.   2797.  1855.     3263. 
2 fairly_active_minu…   627   18.6   15.1   15.6    7.49   26.5    1.50     47.9
3 lightly_active_min…   627  173.    67.0  173.   128.    218.    65.9     283. 
4 sedentary_minutes     627  882.   197.   835.   735.   1020.   618.     1255. 
5 sex_at_birth            0  NaN     NA     NA     NA      NA     NA        NA  
6 steps                 627 6899.  3695.  6513.  4411.   9069.  1495.    12929. 
7 very_active_minutes   627   22.1   21.8   15.7    6.48   29.5    0.650    66.2


In [9]:
table(pca_activity_sex$sex_at_birth)


  Male 
449924 